In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import average_precision_score, precision_recall_curve

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet

In [2]:
df = pd.read_csv("final_combined_data.csv")
df.head()

,Date,Open,High,Low,Close,Volume,Ticker,Return,Volatility,Volume_Zscore,Return_Zscore,Manipulation_Flag,stock,sentiment_score,risk_score,event_count,twitter_sentiment
0,2018-01-02,39.986349,40.489233,39.774854,40.479832,102223600,AAPL,0.000000,0.014841,-0.092570,0.000000,0,AAPL,0.5,1.0,0.0,0.5
1,2018-01-03,40.543277,41.017963,40.409333,40.472778,118071600,AAPL,-0.000174,0.014841,0.194878,-0.070361,0,AAPL,0.5,1.0,0.0,0.5
2,2018-01-04,40.545623,40.764168,40.437528,40.660770,89738400,AAPL,0.004645,0.014841,-0.319025,0.171144,0,AAPL,0.5,1.0,0.0,0.5
3,2018-01-05,40.757138,41.210672,40.665491,41.123726,94640000,AAPL,0.011386,0.014841,-0.230121,0.508955,0,AAPL,0.5,1.0,0.0,0.5
4,2018-01-08,40.970982,41.267071,40.872282,40.970982,82271200,AAPL,-0.003714,0.014841,-0.454464,-0.247764,0,AAPL,0.5,1.0,0.0,0.5


In [3]:
X = df.drop(columns=['Date', 'stock', 'Ticker'])  # drop non-feature columns
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Anomaly Detection Models

### Isolation Forest
A popular unsupervised anomaly detection algorithm that learns from "normal" data and scores anomalies (isolates outliers in data within a forest of random trees)

Reference: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html

In [4]:
iso = IsolationForest(n_estimators=100, contamination=0.005, random_state=42)
iso.fit(X_scaled)
iso_scores = -iso.decision_function(X_scaled)  # negative sign ensures that higher means more anomalous

df['iso_anomaly_score'] = iso_scores
df

,Date,Open,High,Low,Close,Volume,Ticker,Return,Volatility,Volume_Zscore,Return_Zscore,Manipulation_Flag,stock,sentiment_score,risk_score,event_count,twitter_sentiment,iso_anomaly_score
0,2018-01-02,39.986349,40.489233,39.774854,40.479832,102223600,AAPL,0.000000,0.014841,-0.092570,0.000000,0,AAPL,0.5,1.0,0.0,0.5,-0.341268
1,2018-01-03,40.543277,41.017963,40.409333,40.472778,118071600,AAPL,-0.000174,0.014841,0.194878,-0.070361,0,AAPL,0.5,1.0,0.0,0.5,-0.332834
2,2018-01-04,40.545623,40.764168,40.437528,40.660770,89738400,AAPL,0.004645,0.014841,-0.319025,0.171144,0,AAPL,0.5,1.0,0.0,0.5,-0.348757
3,2018-01-05,40.757138,41.210672,40.665491,41.123726,94640000,AAPL,0.011386,0.014841,-0.230121,0.508955,0,AAPL,0.5,1.0,0.0,0.5,-0.339562
4,2018-01-08,40.970982,41.267071,40.872282,40.970982,82271200,AAPL,-0.003714,0.014841,-0.454464,-0.247764,0,AAPL,0.5,1.0,0.0,0.5,-0.344803
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12291,2023-12-22,0.211000,0.220000,0.210000,0.217600,4682395,ZOM,0.003690,0.076343,-0.300652,0.032187,0,ZOM,0.5,1.0,0.0,0.5,-0.355097
12292,2023-12-26,0.216800,0.228500,0.214500,0.221900,9352873,ZOM,0.019761,0.072518,-0.210634,0.240736,0,ZOM,0.5,1.0,0.0,0.5,-0.354489
12293,2023-12-27,0.222500,0.228400,0.211000,0.215000,5095105,ZOM,-0.031095,0.078142,-0.292697,-0.419208,0,ZOM,0.5,1.0,0.0,0.5,-0.355147
12294,2023-12-28,0.216000,0.218200,0.207000,0.210000,4325280,ZOM,-0.023256,0.066950,-0.307534,-0.317481,0,ZOM,0.5,1.0,0.0,0.5,-0.356488


### Autoencoder-based
A type of neural network specialized in learning how to compress input data into a smaller representation and then reconstruct it back to the original form. I applied unsupervised anomaly detection using an autoencoder neural network to identify potential stock manipulation events without relying on labeled ground truth data. 
 - When trained on mostly normal data, the autoencoder learns the typical patterns and relationships among features like return, volatility, risk scores, and sentiment.
 - When given a new data point, the model tries to reconstruct it from its learned representation.
 - If the new point is similar to the normal patterns, it will reconstruct well with a low reconstruction error.
 - If the point is very different (anomalous), the autoencoder will struggle to reconstruct it accurately, resulting in a high reconstruction error.

 Reference: https://keras.io/examples/timeseries/timeseries_anomaly_detection/

In [5]:
input_dim = X_scaled.shape[1]

# a feedforward neural network with layers that compress and decompress the feature space.
autoencoder = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(input_dim, activation='linear')
])
autoencoder.compile(optimizer='adam', loss='mse')

#Training the model using all available data, as the majority of days reflect normal trading behavior.
autoencoder.fit(X_scaled, X_scaled, epochs=50, batch_size=256, validation_split=0.1, verbose=0)

reconstructions = autoencoder.predict(X_scaled)
reconstruction_errors = np.mean(np.square(X_scaled - reconstructions), axis=1) # average squared difference between each day’s input and its reconstruction

df['autoencoder_error'] = reconstruction_errors

# Ranking all stock-days by their reconstruction errors
top_anomalies = df.sort_values('autoencoder_error', ascending=False).head(10)
print(top_anomalies[['Date', 'stock', 'autoencoder_error', 'iso_anomaly_score']])

385/385 ━━━━━━━━━━━━━━━━━━━━ 0s 420us/step
             Date stock  autoencoder_error  iso_anomaly_score
11549  2021-01-12   ZOM           0.163627           0.048547
2281   2021-01-27   AMC           0.135981           0.030853
3882   2021-06-09   GME           0.101116          -0.122197
9895   2020-06-16  TSLA           0.099281          -0.037041
11416  2020-07-02   ZOM           0.099187          -0.065668
11544  2021-01-05   ZOM           0.094243           0.001944
4753   2021-11-17  GOEV           0.088893           0.011171
11377  2020-05-07   ZOM           0.085192          -0.031374
5452   2018-08-30  PLUG           0.077895          -0.227141
9005   2022-11-29  TLRY           0.075776          -0.084525


In [6]:
# Create weak labels: top 1% anomalies = 1
threshold = np.percentile(df['autoencoder_error'], 99)
df['pseudo_label'] = (df['autoencoder_error'] >= threshold).astype(int)
df


,Date,Open,High,Low,Close,Volume,Ticker,Return,Volatility,Volume_Zscore,Return_Zscore,Manipulation_Flag,stock,sentiment_score,risk_score,event_count,twitter_sentiment,iso_anomaly_score,autoencoder_error,pseudo_label
0,2018-01-02,39.986349,40.489233,39.774854,40.479832,102223600,AAPL,0.000000,0.014841,-0.092570,0.000000,0,AAPL,0.5,1.0,0.0,0.5,-0.341268,0.000165,0
1,2018-01-03,40.543277,41.017963,40.409333,40.472778,118071600,AAPL,-0.000174,0.014841,0.194878,-0.070361,0,AAPL,0.5,1.0,0.0,0.5,-0.332834,0.000198,0
2,2018-01-04,40.545623,40.764168,40.437528,40.660770,89738400,AAPL,0.004645,0.014841,-0.319025,0.171144,0,AAPL,0.5,1.0,0.0,0.5,-0.348757,0.000212,0
3,2018-01-05,40.757138,41.210672,40.665491,41.123726,94640000,AAPL,0.011386,0.014841,-0.230121,0.508955,0,AAPL,0.5,1.0,0.0,0.5,-0.339562,0.000272,0
4,2018-01-08,40.970982,41.267071,40.872282,40.970982,82271200,AAPL,-0.003714,0.014841,-0.454464,-0.247764,0,AAPL,0.5,1.0,0.0,0.5,-0.344803,0.000249,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12291,2023-12-22,0.211000,0.220000,0.210000,0.217600,4682395,ZOM,0.003690,0.076343,-0.300652,0.032187,0,ZOM,0.5,1.0,0.0,0.5,-0.355097,0.000252,0
12292,2023-12-26,0.216800,0.228500,0.214500,0.221900,9352873,ZOM,0.019761,0.072518,-0.210634,0.240736,0,ZOM,0.5,1.0,0.0,0.5,-0.354489,0.000143,0
12293,2023-12-27,0.222500,0.228400,0.211000,0.215000,5095105,ZOM,-0.031095,0.078142,-0.292697,-0.419208,0,ZOM,0.5,1.0,0.0,0.5,-0.355147,0.000106,0
12294,2023-12-28,0.216000,0.218200,0.207000,0.210000,4325280,ZOM,-0.023256,0.066950,-0.307534,-0.317481,0,ZOM,0.5,1.0,0.0,0.5,-0.356488,0.000087,0


In [7]:
df['pseudo_label'].value_counts()

pseudo_label
0    12173
1      123
Name: count, dtype: int64

## Classification

### Random Forest Classifier
We use anomaly-based pseudo-labels to train RandomForest classifier

In [8]:
X_supervised = X_scaled
y_supervised = df['pseudo_label']

clf = RandomForestClassifier(class_weight='balanced')
clf.fit(X_supervised, y_supervised)

y_scores = clf.predict_proba(X_supervised)[:,1]
pr_auc = average_precision_score(y_supervised, y_scores)
print(f"Random Forest PR-AUC on pseudo-labels: {pr_auc:.3f}")

y_pred = clf.predict(X_supervised)
print(classification_report(y_supervised, y_pred))

Random Forest PR-AUC on pseudo-labels: 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     12173
           1       1.00      1.00      1.00       123

    accuracy                           1.00     12296
   macro avg       1.00      1.00      1.00     12296
weighted avg       1.00      1.00      1.00     12296



###  Transformer-Based Models for Time-Series